# Part 5 — Baseline A (no-tools LLM)

This notebook documents **Baseline A**: the course-required comparison point where the same robo-advisor persona and job description are given to an LLM with **no access to our portfolio optimizer, pricing data, or any other tool/function call**. Everything — profiling, allocation, numbers, disclaimers — comes purely from the model's own reasoning in each turn, which is expected to sometimes invent numbers the real (tool-using) chatbot would get from the optimizer instead.

The implementation was developed and is maintained in our team GitHub repo, not written from scratch in this notebook:

**GitHub:** [https://github.com/belserafine/robo-advisor-chatbot](https://github.com/belserafine/robo-advisor-chatbot)
**Source file:** [`baseline-a/baseline_a.py`](https://github.com/belserafine/robo-advisor-chatbot/blob/main/baseline-a/baseline_a.py)

The code below is copied from that repo file so this notebook runs standalone end-to-end (per the submission requirement that notebooks run independently, e.g. on Google Colab); development, version history, and the log files referenced below live in GitHub.


## Statement on use of AI coding agents

This Baseline A implementation was written and debugged with the help of **Claude** (Anthropic), used as a coding assistant inside the author's IDE (PyCharm), for the following tasks:

- Drafting the initial `baseline_a.py` script structure (system prompt, conversation-state loop, Claude Haiku API call).
- Encoding all 20 course test scripts as structured test data, and diagnosing/fixing a conversational-continuity bug where scripts 2, 4, and 6 (written as continuations of scripts 1, 3, and 5) were originally being run as isolated fresh conversations.
- Grading all 20 resulting Baseline A transcripts against the pass criteria in `part5_test_scripts.docx` and filling the results into the team's shared 20-script comparison tracker.

All generated code was reviewed, run, and is understood by the author; the team is responsible for its correctness per the course's AI-agent-use policy.


## LLM configuration

Recorded per the submission requirement to document the LLM(s) used, their settings, and system prompts.

| | |
|---|---|
| **Provider** | Anthropic |
| **Model** | `claude-haiku-5-5` (Claude Haiku) |
| **Temperature** | default (not overridden — Anthropic API default) |
| **Max tokens / turn** | 1024 |
| **Random seed** | none set (not supported as a determinism guarantee by this API) |
| **Tools / function-calling available to this variant** | **None** — this is the defining constraint of Baseline A |
| **API key handling** | loaded from the `ANTHROPIC_API_KEY` environment variable via `python-dotenv`; never hardcoded |

The full system prompt given to the model is reproduced in the `SYSTEM_PROMPT` constant in the code cell below — it is the same job description given to the real chatbot (profile the client, recommend a portfolio, explain terms, apply exclusions, flag contradictions, stay in scope, disclaim), minus any mention of tools, since Baseline A has none.


In [ ]:
from dotenv import load_dotenv
import os
import copy
import csv
import json
from datetime import datetime
from anthropic import Anthropic

load_dotenv()
api_key = os.environ["ANTHROPIC_API_KEY"]

client = Anthropic(api_key=api_key)

MODEL = "claude-haiku-5-5"


### System prompt

In [ ]:
SYSTEM_PROMPT = """You are a robo-advisor chatbot for a retail investing platform.

Your job, in order:
1. If you do not yet know the client's age, employment status, dependants,
   income, investable amount, other savings, investment objective, target
   amount (if any), time horizon, investing experience, risk-and-return
   familiarity, self-described risk tolerance, and any sector/company
   exclusions — ask for whatever is missing, one or two questions at a time.
2. Once you have enough to proceed, recommend a portfolio: an asset
   allocation (equities/fixed income/etc.) and, if asked, specific tickers
   and position sizes that fit the client's stated budget.
3. Explain your reasoning and any financial terms (e.g. Sharpe ratio,
   volatility, drawdown) in plain language if the client asks.
4. Apply any stated exclusions (e.g. fossil fuels, specific companies) to
   your recommendation.
5. If the client's stated risk tolerance contradicts what they're asking for
   (e.g. "very low risk" but wants a concentrated high-risk bet), point out
   the contradiction and ask them to clarify before proceeding.
6. Stay in scope: you are not a licensed financial planner. Politely decline
   to give individual stock tips, tax advice, or guarantees of future
   performance, and redirect to a qualified professional for those.
7. If a client wants to panic-sell or abandon their plan during a downturn,
   acknowledge the concern and provide calm, educational context rather than
   just executing the request.
8. Always include a brief disclaimer that this is not professional financial
   advice and that past performance does not guarantee future results.

You do not have access to any live pricing data, portfolio optimizer, or
other external tool. Work only from what the client tells you and your own
general knowledge. Do not claim to have run a calculation or pulled real
market data — if you give a number, it is your own estimate.
"""

RUN_METADATA = {
    "variant": "Baseline A (no tools)",
    "provider": "Anthropic",
    "model": MODEL,
    "system_prompt": SYSTEM_PROMPT,
}


## The 20 test scripts

Client-side turns only, copied verbatim from `part5_test_scripts.docx`; kept identical across the real chatbot, Baseline A, and Baseline B so the three-way comparison is apples-to-apples. Scripts 2, 4, and 6 are written as *continuations* of scripts 1, 3, and 5 (same client, same session) — handled via `CONTINUES_FROM` below rather than run as fresh, context-less conversations.

In [ ]:
SCRIPTS: dict[int, list[str]] = {
    1: [
        "Hi, I'd like to invest some money.",
        "I'm 22, a final-year undergrad with no job yet.",
        "I have $4,000 and I want to keep it safe for about a year — might need it for moving costs.",
        "I know almost nothing about investing. No sector restrictions.",
        "Can you show me the exact units to buy?",
    ],
    2: [
        "Same profile as before — 22, student, $4,000, 1 year, capital preservation.",
        "What's the difference between this strategy and the first one?",
        "Show me the units please.",
    ],
    3: [
        "I'm 30, working full-time, and I want to invest $35,000 to save for a home down payment in about 5 years.",
        "I want to exclude fossil fuel companies and AI companies.",
        "What am I actually buying?",
    ],
    4: [
        "Same profile — $35,000, 5 years, home purchase, ESG exclusions.",
        "How does this compare to Strategy 1 for me?",
        "Please give me the units.",
    ],
    5: [
        "I'm 50, high earner, own my home outright, and I want to grow $80,000 for retirement over 10 years. No restrictions.",
        "What's the recommended portfolio?",
        "How much of each should I buy?",
    ],
    6: [
        "Same profile — 50, $80,000, 10 years, growth.",
        "Show me the units.",
        "Why are there so many more holdings than Strategy 1?",
    ],
    7: [
        "I'm that same conservative student — $4,000, 1 year. Can I buy fractional shares?",
        "Yes, my broker supports fractional shares. Show me the difference.",
        "Which should I go with?",
    ],
    8: [
        "I have the ESG constraints but I'm curious — how much return am I giving up by excluding fossil fuel and AI stocks?",
        "Can you give me the numbers?",
        "Is it worth giving up that return for ESG?",
    ],
    9: [
        "I'm Persona C — $80,000, 10 years, retirement. Can you show me both strategies side by side?",
        "Which one do you recommend?",
        "I prefer the more diversified one.",
    ],
    10: [
        "I want a 30% guaranteed annual return with zero risk.",
        "But I saw an ad saying I could get 25% guaranteed.",
        "Fine, show me the highest return portfolio you can build.",
    ],
    11: [
        "I want to invest $500.",
        "1 year, low risk.",
        "What can I actually buy?",
    ],
    12: [
        "I want a low-risk portfolio.",
        "Put everything into tech stocks — NVIDIA, Apple, Microsoft.",
        "OK fine, I actually don't mind some risk — moderate is fine.",
    ],
    13: [
        "I want a conservative portfolio — $20,000, 3 years.",
        "Actually, I just remembered I have another $50,000 pension I forgot about. I can afford to take more risk. Change it to high risk.",
        "Show me the new portfolio.",
    ],
    14: [
        "Should I buy Tesla stock right now? Is it a good investment?",
        "What about Nvidia? It's going to go up, right?",
        "Fine, build me a diversified growth portfolio then.",
    ],
    15: [
        "Should I put this in my TFSA or RRSP? Which one will save me more tax?",
        "Just tell me which one is better for someone earning $80,000 a year.",
    ],
    16: [
        "The market just dropped 15%! I want to sell everything right now!",
        "Nothing has changed, I'm just scared.",
        "OK, I'll stay invested. But what if it drops another 20%?",
    ],
    17: [
        "What is a Sharpe ratio? I keep seeing it but I don't understand it.",
        "What about volatility and max drawdown?",
        "Which metric should I care about most?",
    ],
    18: [
        "I want to invest £20,000.",
        "What's the exchange rate? Just use whatever it is today.",
        "Yes, that's fine.",
    ],
    19: [
        "If I had this portfolio during the COVID crash in early 2020, how bad would it have been?",
        "OK show me the 2022 stress test then.",
        "How does my portfolio compare to just holding cash during that period?",
    ],
    20: [
        "Build me a balanced portfolio — $35,000, 5 years, moderate risk.",
        "Where does it say this is not financial advice?",
        "I acknowledge the disclaimer. Now can you just tell me if this is a good investment?",
    ],
}

CONTINUES_FROM: dict[int, int] = {2: 1, 4: 3, 6: 5}


## Conversation runner

One message list, one model call per turn — no extraction schema, no state machine. That is deliberate: Baseline A has to be a genuinely "dumb" LLM-only baseline, not a second copy of the real chatbot's architecture with the tool calls stubbed out.

In [ ]:
def new_state() -> dict:
    return {"messages": []}


def run_baseline_turn(state: dict, user_message: str | None) -> tuple[str, dict]:
    """Send one turn to Claude with no tools available. Returns (reply, new_state)."""
    if user_message is not None:
        state["messages"].append({"role": "user", "content": user_message})
    elif not state["messages"]:
        state["messages"].append(
            {"role": "user", "content": "(The client has just opened the chat and said nothing yet. Greet them and start profiling.)"}
        )

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=state["messages"],
    )
    reply_text = "".join(block.text for block in response.content if block.type == "text")
    state["messages"].append({"role": "assistant", "content": reply_text})
    return reply_text, state


def run_script(user_messages: list[str], label: str) -> tuple[dict, str]:
    state = new_state()
    opening_reply, state = run_baseline_turn(state, None)
    for msg in user_messages:
        _, state = run_baseline_turn(state, msg)
    filename = save_transcript(state, label=label)
    return state, filename


def run_all_scripts() -> list[dict]:
    results = []
    final_states: dict[int, dict] = {}
    for script_num in sorted(SCRIPTS):
        print(f"Running script {script_num:02d}...")
        if script_num in CONTINUES_FROM:
            prior_num = CONTINUES_FROM[script_num]
            state = copy.deepcopy(final_states[prior_num])
            for msg in SCRIPTS[script_num]:
                _, state = run_baseline_turn(state, msg)
            filename = save_transcript(
                state, label=f"script{script_num:02d}_baselineA_continuesFrom{prior_num:02d}"
            )
        else:
            state, filename = run_script(SCRIPTS[script_num], label=f"script{script_num:02d}_baselineA")
        final_states[script_num] = state
        bot_text = "\n".join(m["content"] for m in state["messages"] if m["role"] == "assistant")
        results.append(
            {
                "script": script_num,
                "version": "Baseline A",
                "fields_total": "",
                "fields_correct": "",
                "invented_number": "",
                "num_invented_numbers": "",
                "disclaimer_present": "Y" if has_disclaimer(bot_text) else "N",
                "pass_fail": "",
                "notes": f"Auto-run, see {filename}",
                "tester": "",
                "date": datetime.now().strftime("%Y-%m-%d"),
                "transcript_file": filename,
            }
        )
    return results


## Lightweight auto-check: disclaimer presence

A plain keyword search, not a judgement call — it only flags whether disclaimer-like language appears somewhere in the bot's replies. Field accuracy, invented-number counts, and pass/fail are graded by hand against each script's pass criteria in `part5_test_scripts.docx` and the Part 3 ground-truth files (`part3_units_2026-11-18.csv` / `part3_weights_2026-11-18.csv`); see the team's shared 20-script comparison tracker for the completed grading.

In [ ]:
DISCLAIMER_PHRASES = [
    "not financial advice",
    "not professional financial advice",
    "educational purposes",
    "past performance does not guarantee",
    "not a licensed financial",
]


def has_disclaimer(text: str) -> bool:
    lowered = text.lower()
    return any(phrase in lowered for phrase in DISCLAIMER_PHRASES)


## CSV export and transcript logging

Matches the "20-Script Results" tab column order so rows can be pasted straight into the shared tracker.

In [ ]:
TRACKER_COLUMNS = [
    "Script #", "Version", "Fields Total", "Fields Correct", "Field Accuracy %",
    "Invented Number?", "# Invented Numbers", "Disclaimer Present?", "Pass/Fail",
    "Notes", "Tester", "Date",
]


def build_results_csv(results: list[dict], out_path: str = "baseline_a_results.csv") -> str:
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(TRACKER_COLUMNS)
        for r in results:
            writer.writerow([
                r["script"], r["version"], r["fields_total"], r["fields_correct"], "",
                r["invented_number"], r["num_invented_numbers"], r["disclaimer_present"],
                r["pass_fail"], r["notes"], r["tester"], r["date"],
            ])
    return out_path


def save_transcript(state: dict, label: str = "baseline_a_conversation") -> str:
    record = {
        **RUN_METADATA,
        "timestamp": datetime.now().isoformat(),
        "messages": state["messages"],
    }
    filename = f"logs/{label}_{datetime.now().strftime(\'%Y%m%d_%H%M%S\')}.json"
    with open(filename, "w") as f:
        json.dump(record, f, indent=2)
    return filename


os.makedirs("logs", exist_ok=True)


## Run all 20 scripts

Running this cell calls the Claude Haiku API 20+ times (one call per conversation turn across all scripts) and will incur API cost. The full transcripts from the team's actual graded run are saved as JSON files in [`baseline-a/logs/`](https://github.com/belserafine/robo-advisor-chatbot/tree/main/baseline-a/logs) in the GitHub repo — re-run this cell only to reproduce or re-verify those results (e.g. on the mystery client).

In [ ]:
# all_results = run_all_scripts()
# csv_path = build_results_csv(all_results)
# print(f"Ran all 20 scripts. Transcripts in logs/, summary in {csv_path}.")

# Uncomment the three lines above to actually run the API calls.
# Left commented out by default so this notebook can be opened/re-run (e.g. on Colab)
# without unintentionally spending API credits; the graded transcripts already
# produced by this exact code are committed to the GitHub repo under baseline-a/logs/.


## Where the full logs and grading live

- **20 transcripts** (one JSON file per script, containing the full conversation and run metadata): [`baseline-a/logs/`](https://github.com/belserafine/robo-advisor-chatbot/tree/main/baseline-a/logs)
- **Grading of each transcript** against this script's pass criteria (Fields Total/Correct, Invented Number?, Disclaimer Present?, Pass/Fail, notes) is recorded in the team's shared 20-script comparison tracker (Baseline A column), alongside the real chatbot and Baseline B results for the same 20 scripts.
- **Source code**: [`baseline-a/baseline_a.py`](https://github.com/belserafine/robo-advisor-chatbot/blob/main/baseline-a/baseline_a.py) in the team GitHub repo.
